# OmniVoice Studio — Kaggle Gradio Full Studio

This notebook keeps the **Gradio-first** workflow, but upgrades it to the same production runtime contract used by the maintained Kaggle Studio notebook.

```text
cuda:0  -> OmniVoice TTS
cuda:1  -> Whisper ASR verification when a second T4 exists
CPU/RAM -> preprocessing, Gradio/FastAPI/MCP, file I/O
SSD     -> /kaggle/working/OmniVoiceStudio + runtime-local caches
```

Two launch modes are supported:

- `unified` (default): full OmniVoice Studio surface, with Gradio at `/ui` plus REST, SSE jobs, OpenAPI and MCP over an authenticated temporary Cloudflare URL.
- `gradio-share`: full interactive Studio UI through Gradio's temporary share URL, without REST/MCP.

The source package and model cache are bound to exact immutable revisions. Do not publish synthetic startup timings.


In [ ]:
# Resolve an immutable OmniVoice source revision first.
import json
import os
import re
import sys
import urllib.request

PACKAGE_REF = os.environ.get("OMNIVOICE_PACKAGE_REF", "").strip().lower()
if PACKAGE_REF:
    if re.fullmatch(r"[0-9a-f]{40}", PACKAGE_REF) is None:
        raise RuntimeError("OMNIVOICE_PACKAGE_REF must be an exact 40-character commit SHA.")
else:
    try:
        with urllib.request.urlopen(
            "https://api.github.com/repos/binhminhanh1235/OmniVoice/branches/master",
            timeout=20,
        ) as response:
            PACKAGE_REF = str(json.load(response)["commit"]["sha"]).lower()
    except Exception as exc:
        raise RuntimeError(
            "Cannot resolve current OmniVoice master exactly. Enable Kaggle Internet or "
            "set OMNIVOICE_PACKAGE_REF to a verified 40-character commit SHA."
        ) from exc
    if re.fullmatch(r"[0-9a-f]{40}", PACKAGE_REF) is None:
        raise RuntimeError("GitHub returned a non-immutable package revision.")

BOOTSTRAP_URL = (
    "https://raw.githubusercontent.com/binhminhanh1235/OmniVoice/"
    f"{PACKAGE_REF}/notebooks/hosted_runtime_bootstrap.py"
)
try:
    with urllib.request.urlopen(BOOTSTRAP_URL, timeout=30) as response:
        bootstrap_source = response.read().decode("utf-8")
except Exception as exc:
    raise RuntimeError(
        f"Cannot load hosted-runtime bootstrap from exact revision {PACKAGE_REF}."
    ) from exc

exec(compile(bootstrap_source, BOOTSTRAP_URL, "exec"), globals(), globals())
globals().update(bootstrap_hosted_runtime(PACKAGE_REF))
del bootstrap_source

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
import importlib
import shutil
import torch
import omnivoice.hardware_quality as hardware_quality
from omnivoice.runtime_workspace import detect_runtime_workspace, ensure_runtime_workspace

hardware_quality = importlib.reload(hardware_quality)
runtime = ensure_runtime_workspace(detect_runtime_workspace())
if runtime.environment != "kaggle":
    raise RuntimeError(f"Expected Kaggle runtime, detected: {runtime.environment}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle Notebook settings. T4 x2 is preferred.")

GPU_COUNT = torch.cuda.device_count()
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GiB")

hardware = hardware_quality.detect_hardware(device_index=0)
TTS_DEVICE = "cuda:0"
ASR_DEVICE = "cuda:1" if GPU_COUNT >= 2 else hardware.recommended_asr_device

if GPU_COUNT >= 2 and hardware.recommended_asr_device != "cuda:1":
    print("WARNING: forcing ASR to cuda:1 so the second T4 is used.")

quality_store = hardware_quality.HardwareQualitySettingsStore(WORKSPACE)
if not quality_store.path.exists():
    quality_store.set_default(hardware.recommended_preset)
current_preset = quality_store.load().default_preset

print("Package ref:", PACKAGE_REF)
print("Runtime:", runtime.summary())
print("Hardware:", hardware.summary())
print("OmniVoice device:", TTS_DEVICE)
print("Whisper ASR device:", ASR_DEVICE)
print("Whisper ASR model:", ASR_MODEL, "@", ASR_MODEL_REVISION)
print("Workspace quality preset:", current_preset)
print("Workspace:", WORKSPACE)
print("Startup evidence:", STARTUP_CACHE_EVIDENCE)
print("Cache export:", CACHE_EXPORT_BASE)
usage = shutil.disk_usage("/kaggle/working")
print(f"Local SSD free: {usage.free / 1024**3:.1f} GiB")


## Studio feature surface

The Gradio UI is built by the same `project_studio_voice_doctor.build_demo()` used by the unified server. It includes:

- Projects / Workspace;
- Text Doctor;
- Section History;
- Queue;
- Pause / Resume;
- Voice Library / Voice Doctor;
- Hardware & Quality;
- Advanced settings;
- Export;
- Storage & Backup;
- audio download controls.

`unified` mode mounts that same Gradio app at `/ui` and also enables `/api/v1`, durable jobs + SSE, `/docs`, `/mcp`, and `/health`.

For a single-GPU runtime, ASR may fall back to CPU. In that case Lazy CPU ASR defers ASR construction until the first real transcription/verification request. On dual-T4, explicit `cuda:1` ASR remains eager.


In [ ]:
# Choose the public surface.
LAUNCH_MODE = "unified"  # "unified" or "gradio-share"
ALLOW_INSECURE_PUBLIC = False  # only for an intentional temporary test
SERVER_PORT = 8000

if LAUNCH_MODE not in {"unified", "gradio-share"}:
    raise ValueError("LAUNCH_MODE must be 'unified' or 'gradio-share'.")

print("Launch mode:", LAUNCH_MODE)
if LAUNCH_MODE == "unified":
    print("Full surface: Gradio /ui + REST + SSE jobs + OpenAPI + MCP + health.")
    if ALLOW_INSECURE_PUBLIC:
        print("WARNING: public endpoint will be intentionally unauthenticated.")
    else:
        print(
            "Secure mode requires Kaggle Secrets: OMNIVOICE_API_TOKEN, "
            "OMNIVOICE_UI_USERNAME, OMNIVOICE_UI_PASSWORD."
        )
else:
    print("Gradio UI-only mode. REST/MCP are intentionally not exposed.")


## Production cold/warm acceptance

The bootstrap has already produced `startup-cache-evidence.json`.

For a genuine cold run set `ACCEPTANCE_SAMPLE = "cold"`, execute the cell, then save/version `/kaggle/working/OmniVoiceStartupCache` as the Kaggle Dataset `omnivoice-startup-cache`.

Start a fresh Kaggle session with that Dataset attached at `/kaggle/input/omnivoice-startup-cache`, then set `ACCEPTANCE_SAMPLE = "warm"` and execute the same cell.

When both samples exist for the same exact package revision, the notebook runs the exact-revision acceptance checker automatically.


In [ ]:
from omnivoice.lazy_asr import should_defer_asr_startup

evidence = json.loads(Path(STARTUP_CACHE_EVIDENCE).read_text(encoding="utf-8"))
if evidence.get("package_ref") != PACKAGE_REF:
    raise RuntimeError("Startup evidence package_ref does not match PACKAGE_REF.")

print("Bootstrap seconds:", evidence.get("bootstrap_seconds"))
print("Resource fast path:", evidence.get("resource_fast_path"))
print("Wheel fast path:", evidence.get("wheel_fast_path"))

lazy_cpu_expected = should_defer_asr_startup(ASR_DEVICE)
print("ASR startup policy:", "lazy CPU" if lazy_cpu_expected else "explicit accelerator/eager")

ACCEPTANCE_SAMPLE = ""  # set to "cold" or "warm" only for a genuine run
sample = ACCEPTANCE_SAMPLE.strip().lower()
if sample not in {"", "cold", "warm"}:
    raise ValueError("ACCEPTANCE_SAMPLE must be empty, 'cold', or 'warm'.")

acceptance_root = Path(CACHE_EXPORT_BASE) / "acceptance" / PACKAGE_REF
acceptance_root.mkdir(parents=True, exist_ok=True)

if CACHE_SOURCE_BASE is not None:
    source_acceptance = Path(CACHE_SOURCE_BASE) / "acceptance" / PACKAGE_REF
    if source_acceptance.is_dir():
        shutil.copytree(source_acceptance, acceptance_root, dirs_exist_ok=True)

if sample:
    sample_path = acceptance_root / f"{sample}.json"
    shutil.copy2(STARTUP_CACHE_EVIDENCE, sample_path)
    print("Recorded acceptance sample:", sample_path)

cold_path = acceptance_root / "cold.json"
warm_path = acceptance_root / "warm.json"
if cold_path.is_file() and warm_path.is_file():
    acceptance_url = (
        "https://raw.githubusercontent.com/binhminhanh1235/OmniVoice/"
        f"{PACKAGE_REF}/scripts/hosted_cache_acceptance.py"
    )
    acceptance_script = Path("/kaggle/working/hosted_cache_acceptance.py")
    with urllib.request.urlopen(acceptance_url, timeout=30) as response:
        acceptance_script.write_bytes(response.read())
    subprocess.run(
        [sys.executable, str(acceptance_script), str(cold_path), str(warm_path)],
        check=True,
    )
else:
    print("Cold/warm pair not complete yet. Record only genuine samples.")

print("Acceptance evidence directory:", acceptance_root)


## Launch OmniVoice Studio

### Default: `unified`

The notebook starts a temporary Cloudflare Quick Tunnel first, discovers its public URL, configures the Studio security policy for that exact host, then launches one OmniVoice process serving:

```text
<public-url>/ui       Gradio Studio
<public-url>/api/v1   REST API
<public-url>/docs     OpenAPI
<public-url>/mcp      MCP
<public-url>/health   health/capabilities
```

Secure mode reads credentials only from Kaggle Secrets. If you deliberately want a disposable unauthenticated test, set `ALLOW_INSECURE_PUBLIC = True`.

### Alternative: `gradio-share`

This launches the same complete interactive Studio UI with Gradio `share=True`, but intentionally omits REST/MCP.


In [ ]:
import queue
import stat
import subprocess
import threading
import time

def _kaggle_secret(name: str) -> str:
    from kaggle_secrets import UserSecretsClient

    try:
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    return str(value or "").strip()

def _configure_public_auth() -> None:
    if ALLOW_INSECURE_PUBLIC:
        os.environ["OMNIVOICE_ALLOW_INSECURE_PUBLIC"] = "1"
        os.environ.pop("OMNIVOICE_API_TOKEN", None)
        os.environ.pop("OMNIVOICE_UI_USERNAME", None)
        os.environ.pop("OMNIVOICE_UI_PASSWORD", None)
        return

    required_names = (
        "OMNIVOICE_API_TOKEN",
        "OMNIVOICE_UI_USERNAME",
        "OMNIVOICE_UI_PASSWORD",
    )
    values = {name: _kaggle_secret(name) for name in required_names}
    missing = [name for name, value in values.items() if not value]
    if missing:
        raise RuntimeError(
            "Missing Kaggle Secrets: "
            + ", ".join(missing)
            + ". Add them in Kaggle > Add-ons > Secrets, or set "
            "ALLOW_INSECURE_PUBLIC=True only for an intentional temporary test."
        )
    for name, value in values.items():
        os.environ[name] = value
    os.environ["OMNIVOICE_API_TOKEN_SCOPES"] = (
        "omnivoice:read,omnivoice:generate,omnivoice:queue,omnivoice:mcp"
    )
    os.environ.pop("OMNIVOICE_ALLOW_INSECURE_PUBLIC", None)

def _ensure_cloudflared() -> Path:
    binary = Path("/kaggle/working/cloudflared")
    if not binary.is_file():
        url = (
            "https://github.com/cloudflare/cloudflared/releases/latest/download/"
            "cloudflared-linux-amd64"
        )
        print("Downloading cloudflared...")
        urllib.request.urlretrieve(url, binary)
        binary.chmod(binary.stat().st_mode | stat.S_IXUSR)
    return binary

def _start_quick_tunnel(port: int):
    binary = _ensure_cloudflared()
    process = subprocess.Popen(
        [
            str(binary),
            "tunnel",
            "--url",
            f"http://127.0.0.1:{port}",
            "--no-autoupdate",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = queue.Queue()

    def _pump():
        assert process.stdout is not None
        for line in process.stdout:
            lines.put(line)

    threading.Thread(target=_pump, daemon=True).start()
    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    deadline = time.monotonic() + 60.0
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(
                f"cloudflared exited before publishing a URL (code {process.returncode})."
            )
        try:
            line = lines.get(timeout=1.0)
        except queue.Empty:
            continue
        print(line.rstrip())
        match = pattern.search(line)
        if match:
            return process, match.group(0)

    process.terminate()
    raise RuntimeError("Timed out while waiting for the temporary Cloudflare URL.")

if LAUNCH_MODE == "gradio-share":
    subprocess.run(
        [
            "omnivoice-project-studio",
            "--model",
            MODEL_ID,
            "--device",
            TTS_DEVICE,
            "--workspace",
            WORKSPACE,
            "--asr-model",
            ASR_MODEL,
            "--asr-device",
            ASR_DEVICE,
            "--share",
        ],
        check=True,
    )
else:
    _configure_public_auth()
    tunnel_process, PUBLIC_URL = _start_quick_tunnel(SERVER_PORT)
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL

    print("\nOmniVoice Studio endpoints")
    print("UI:     ", f"{PUBLIC_URL}/ui")
    print("REST:   ", f"{PUBLIC_URL}/api/v1")
    print("OpenAPI:", f"{PUBLIC_URL}/docs")
    print("MCP:    ", f"{PUBLIC_URL}/mcp")
    print("Health: ", f"{PUBLIC_URL}/health")
    try:
        subprocess.run(
            [
                "omnivoice-studio",
                "serve",
                "--model",
                MODEL_ID,
                "--device",
                TTS_DEVICE,
                "--workspace",
                WORKSPACE,
                "--asr-model",
                ASR_MODEL,
                "--asr-device",
                ASR_DEVICE,
                "--host",
                "0.0.0.0",
                "--port",
                str(SERVER_PORT),
                "--public-url",
                PUBLIC_URL,
            ],
            check=True,
        )
    finally:
        tunnel_process.terminate()
        try:
            tunnel_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            tunnel_process.kill()


## Before ending the Kaggle session

After stopping Studio, run the final cell so runtime cache metadata is persisted to the writable cache export.

Then:

1. save/version `/kaggle/working/OmniVoiceStartupCache` as Kaggle Dataset `omnivoice-startup-cache`;
2. use **Settings → Storage & Backup** if you want project data copied to remote storage;
3. remember that `/kaggle/working/OmniVoiceStudio` itself is ephemeral.


In [ ]:
persist_runtime_cache(CACHE_PREPARATION)
write_workspace_cache_metadata(WORKSPACE, CACHE_PREPARATION)
print("Startup cache export ready:", CACHE_EXPORT_BASE)
print("Workspace is ephemeral:", WORKSPACE)
print("Use Settings > Storage & Backup for persistent project data.")
